# Day 39 — Final Review: Gaps and Edge Cases
**Estimated time:** 2 hours  
**Learning objectives:**
1. Rapidly demonstrate fluency across 40 Python/SQL concepts with mini-examples
2. Identify the 3 concepts most likely to trip you up in an interview
3. Lock in understanding of the trickiest edge cases (NULLs, joins, GROUP BY vs PARTITION BY)

---
> **How to use this notebook:** For each concept, write the one-liner or mini-example *before* running the cell. If you have to look it up, mark it as a gap. Your 3 weakest areas at the end of this notebook are your study targets before any interview.


## 0. Setup

In [ ]:
import pandas as pd, numpy as np, sqlite3
from pathlib import Path

DATA = Path('../datasets')
con = sqlite3.connect(':memory:')
for t, f in [('sales','sales_orders.csv'),('mat','materials_inventory.csv'),
             ('hr','hr_headcount.csv'),('cc','cost_centers.csv') if False else ('cc','cost_center_actuals.csv'),
             ('bw','bw_sales_kpis.csv')]:
    pd.read_csv(DATA/f).to_sql(t, con, if_exists='replace', index=False)

def q(sql): return pd.read_sql_query(sql, con)
print('Ready.')

## PYTHON CONCEPTS (1–20)

In [ ]:
# ── 1. Chaining: read CSV, filter, groupby in one expression ─────────────────
result_1 = (
    # YOUR ONE-LINER
    pd.read_csv(DATA/'sales_orders.csv')
    .query("STATUS == 'OPEN'")
    .groupby('VKORG')['NETWR'].sum()
)
print('1. Chained groupby:', result_1.head(3).to_dict())

In [ ]:
# ── 2. SettingWithCopyWarning — show the WRONG way and the RIGHT way ────────
# WRONG (may not modify original):
df = pd.read_csv(DATA/'sales_orders.csv')
# df[df['STATUS']=='OPEN']['NEW_COL'] = 1   # don't run — this is the wrong way

# RIGHT:
df_open = df[df['STATUS']=='OPEN'].copy()
df_open['NEW_COL'] = 1
print('2. SettingWithCopyWarning avoided: .copy() used')

In [ ]:
# ── 3. NULL behaviour in pandas: NaN vs None vs pd.NA ────────────────────
s = pd.Series([1, None, np.nan, pd.NA, 0])
print('3. isna():', s.isna().tolist())
print('   None == None:', None == None)   # True in Python
print('   nan == nan:', np.nan == np.nan)  # False! Use pd.isna()

In [ ]:
# ── 4. merge vs join: show both ways to join two DataFrames ──────────────────
df1 = pd.DataFrame({'ID':[1,2,3],'A':['x','y','z']})
df2 = pd.DataFrame({'ID':[1,2,4],'B':['p','q','r']})
inner = pd.merge(df1, df2, on='ID', how='inner')
left  = pd.merge(df1, df2, on='ID', how='left')
print('4. inner join rows:', len(inner), ' left join rows:', len(left))

In [ ]:
# ── 5. Join fanout: demonstrate how joining on non-unique key inflates rows ──
df_orders = pd.DataFrame({'KUNNR':['C1','C1','C2'],'NETWR':[100,200,300]})
df_attr   = pd.DataFrame({'KUNNR':['C1','C1','C2'],'ATTR':['a','b','c']})   # 2 rows for C1
merged = df_orders.merge(df_attr, on='KUNNR', how='left')
print('5. Orders:', len(df_orders), '→ After join:', len(merged), '(fanout!)')
print('   Sum NETWR before join:', df_orders['NETWR'].sum())
print('   Sum NETWR after join:', merged['NETWR'].sum(), '(WRONG — inflated)')

In [ ]:
# ── 6. pivot_table vs groupby: when to use each ────────────────────────────
bw = pd.read_csv(DATA/'bw_sales_kpis.csv')

# groupby: more flexible, faster for simple aggregations
gb = bw.groupby(['REGION','SPART'])['REVENUE'].sum()

# pivot_table: produces a 2D table directly, good for reports
pt = bw.pivot_table(values='REVENUE', index='REGION', columns='SPART', aggfunc='sum', fill_value=0)
print('6. pivot_table shape:', pt.shape, '  groupby len:', len(gb))

In [ ]:
# ── 7. apply vs transform: key difference ───────────────────────────────────
df = pd.read_csv(DATA/'sales_orders.csv')

# apply: returns an aggregated result (different shape from input)
agg = df.groupby('VKORG')['NETWR'].apply(sum)

# transform: returns same shape as input (for adding group-level info back to rows)
df['GROUP_TOTAL'] = df.groupby('VKORG')['NETWR'].transform('sum')
print('7. apply shape:', agg.shape, '  transform adds column, df shape:', df.shape)

In [ ]:
# ── 8. Chaining warning: multi-condition boolean indexing ────────────────────
# WRONG: uses 'and' (scalar boolean, breaks on Series)
# df[(df['STATUS'] == 'OPEN') and (df['NETWR'] > 1000)]   # TypeError

# RIGHT: use & with parentheses
correct = df[(df['STATUS'] == 'OPEN') & (df['NETWR'] > 1000)]
print('8. Multi-condition filter rows:', len(correct))

In [ ]:
# ── 9. pd.cut vs pd.qcut ─────────────────────────────────────────────────────
hr = pd.read_csv(DATA/'hr_headcount.csv')

# cut: fixed bin edges (you control where breaks are)
hr['SAL_BAND_CUT'] = pd.cut(hr['SALARY'], bins=[0,60000,100000,200000,9e6],
                             labels=['<60K','60-100K','100-200K','>200K'])

# qcut: quantile-based (equal number of records per bin)
hr['SAL_BAND_QCUT'] = pd.qcut(hr['SALARY'].dropna(), q=4, labels=['Q1','Q2','Q3','Q4'],
                                duplicates='drop')
print('9. cut:', hr['SAL_BAND_CUT'].value_counts().to_dict())

In [ ]:
# ── 10. Memory: float64 vs float32 vs int32 ─────────────────────────────────
import sys
f64 = pd.Series(np.ones(100_000, dtype='float64'))
f32 = pd.Series(np.ones(100_000, dtype='float32'))
print('10. float64:', sys.getsizeof(f64)/1024, 'KB')
print('    float32:', sys.getsizeof(f32)/1024, 'KB  (half the memory)')

In [ ]:
# ── 11. Regex in pandas: extract, str.contains, str.replace ─────────────────
mat = pd.read_csv(DATA/'materials_inventory.csv')
# Extract numeric part of MATNR
mat['MAT_NUM'] = mat['MATNR'].str.extract(r'MAT-(\d+)').astype(int)  # noqa
print('11. Extracted mat num range:', mat['MAT_NUM'].min(), '-', mat['MAT_NUM'].max())

In [ ]:
# ── 12. Date arithmetic edge cases ──────────────────────────────────────────
hr_dates = pd.read_csv(DATA/'hr_headcount.csv', parse_dates=['HIRE_DATE','TERM_DATE'])

# Calculating tenure handles NaT gracefully if you check first
today = pd.Timestamp('2026-04-01')
active = hr_dates[hr_dates['TERM_DATE'].isna()].copy()
active['TENURE_DAYS'] = (today - active['HIRE_DATE']).dt.days

print('12. Avg tenure (active):', active['TENURE_DAYS'].mean() / 365.25, 'years')
print('    NaT subtraction:', pd.NaT - today)  # NaT, doesn't crash

In [ ]:
# ── 13. String methods: .str accessor performance note ────────────────────
import time
df = pd.read_csv(DATA/'sales_orders.csv')
df_big = pd.concat([df]*100)

t0 = time.perf_counter()
_ = df_big['STATUS'].str.upper()
t_str = time.perf_counter() - t0

# Category is faster for .str on low-cardinality columns
df_cat = df_big.copy()
df_cat['STATUS'] = df_cat['STATUS'].astype('category')
t0 = time.perf_counter()
_ = df_cat['STATUS'].str.upper()
t_cat = time.perf_counter() - t0

print(f'13. str.upper on object: {t_str*1000:.1f}ms  on category: {t_cat*1000:.1f}ms')

In [ ]:
# ── 14. stack / unstack / melt / wide_to_long ────────────────────────────────
bw_month = bw.groupby(['REGION','CAL_YEAR_MONTH'])[['REVENUE','COGS']].sum()

# stack: columns → rows
stacked = bw_month.stack().reset_index()
stacked.columns = ['REGION','PERIOD','METRIC','VALUE']
print('14. stack shape:', stacked.shape)

# melt: same idea, on a flat DataFrame
flat = bw.groupby('REGION')[['REVENUE','COGS']].sum().reset_index()
melted = flat.melt(id_vars='REGION', var_name='METRIC', value_name='VALUE')
print('    melt shape:', melted.shape)

In [ ]:
# ── 15. explode() for list-like columns ─────────────────────────────────────
# Useful when BW extracts have comma-separated values in a cell
df_ex = pd.DataFrame({'ID':[1,2],'TAGS':['a,b,c','x,y']})
df_ex['TAGS_LIST'] = df_ex['TAGS'].str.split(',')
exploded = df_ex.explode('TAGS_LIST')
print('15. explode:', exploded.to_string(index=False))

In [ ]:
# ── 16. pd.Grouper for time-based grouping ───────────────────────────────────
so_dates = pd.read_csv(DATA/'sales_orders.csv', parse_dates=['ERDAT'])
so_dates = so_dates.set_index('ERDAT')

# Monthly groupby using pd.Grouper
monthly = so_dates.groupby(pd.Grouper(freq='ME'))['NETWR'].sum()
print('16. Monthly revenue rows:', len(monthly), '  last:', monthly.tail(3).to_dict())

In [ ]:
# ── 17. .pipe() for readable pipeline steps ─────────────────────────────────
def add_inv_value(df):
    df = df.copy()
    df['INV_VALUE'] = df['LABST'] * df['STPRS']
    return df

def flag_dead(df, days=180):
    today = pd.Timestamp('2026-04-01')
    df['DAYS_OLD'] = (today - pd.to_datetime(df['LAST_MOVEMENT_DATE'],errors='coerce')).dt.days.fillna(999)
    df['IS_DEAD'] = df['DAYS_OLD'] > days
    return df

mat_piped = (
    pd.read_csv(DATA/'materials_inventory.csv')
    .pipe(add_inv_value)
    .pipe(flag_dead, days=180)
)
print('17. pipe() result:', mat_piped[['MATNR','INV_VALUE','IS_DEAD']].head(3).to_string(index=False))

In [ ]:
# ── 18. Handling encoding issues in real SAP extracts ────────────────────────
# SAP flat files are often exported in SAP-specific codepages (Latin-1, CP1252)
# pd.read_csv(path, encoding='latin-1')  or  encoding='cp1252'
# For unknown encoding: use chardet
# import chardet
# with open(path,'rb') as f: print(chardet.detect(f.read(100000)))
print('18. Encoding: always try latin-1 or cp1252 for SAP exports that fail UTF-8')

In [ ]:
# ── 19. Type hints in pandas function signatures ─────────────────────────────
from typing import Optional

def filter_open_orders(
    df: pd.DataFrame,
    min_value: float = 0,
    max_age_days: Optional[int] = None
) -> pd.DataFrame:
    """Filter to open orders meeting value and age criteria."""
    result = df[df['STATUS'] == 'OPEN'].copy()
    result = result[result['NETWR'] >= min_value]
    if max_age_days is not None:
        today = pd.Timestamp('2026-04-01')
        ages = (today - pd.to_datetime(result['ERDAT'],errors='coerce')).dt.days
        result = result[ages <= max_age_days]
    return result

so_filtered = filter_open_orders(pd.read_csv(DATA/'sales_orders.csv'), min_value=1000, max_age_days=90)
print('19. Filtered open orders:', len(so_filtered))

In [ ]:
# ── 20. Docstrings: NumPy style (expected in professional code reviews) ──────────────
def calculate_turnover_rate(
    hr_df: pd.DataFrame,
    reference_date: pd.Timestamp,
    lookback_months: int = 12
) -> float:
    """Calculate annualized employee turnover rate.

    Parameters
    ----------
    hr_df : pd.DataFrame
        HR headcount DataFrame with HIRE_DATE, TERM_DATE, PERNR columns.
    reference_date : pd.Timestamp
        The date from which to measure (typically report date).
    lookback_months : int, optional
        Number of months to look back for terminations, by default 12.

    Returns
    -------
    float
        Annualized turnover rate as a percentage (e.g., 15.3 for 15.3%).
    """
    start = reference_date - pd.DateOffset(months=lookback_months)
    hr_p = hr_df.copy()
    hr_p['TERM_DATE'] = pd.to_datetime(hr_p['TERM_DATE'], errors='coerce')
    hr_p['HIRE_DATE'] = pd.to_datetime(hr_p['HIRE_DATE'], errors='coerce')
    terms = hr_p[(hr_p['TERM_DATE'].notna()) & (hr_p['TERM_DATE'] >= start) & (hr_p['TERM_DATE'] <= reference_date)].shape[0]
    active = hr_p[hr_p['TERM_DATE'].isna() | (hr_p['TERM_DATE'] > reference_date)].shape[0]
    return round(terms / active * 100, 1) if active > 0 else 0.0

print('20. Turnover rate:', calculate_turnover_rate(pd.read_csv(DATA/'hr_headcount.csv'), pd.Timestamp('2026-04-01')), '%')

## SQL CONCEPTS (21–40)

In [ ]:
# ── 21. NULL in WHERE: IS NULL vs = NULL ─────────────────────────────────────
# WRONG: WHERE TERM_DATE = NULL   → always returns 0 rows (NULL != anything)
# RIGHT: WHERE TERM_DATE IS NULL

r = q("SELECT COUNT(*) AS n FROM hr WHERE TERM_DATE IS NULL")
print('21. IS NULL:', r.iloc[0]['n'], 'active employees')
r2 = q("SELECT COUNT(*) AS n FROM hr WHERE TERM_DATE = NULL")
print('    = NULL:', r2.iloc[0]['n'], '(always 0!)')

In [ ]:
# ── 22. NULL in aggregates: SUM/AVG ignore NULLs ────────────────────────────
r = q("SELECT AVG(SALARY) AS avg_sal, COUNT(SALARY) AS n_non_null, COUNT(*) AS n_total FROM hr")
print('22. AVG ignores NULLs:', r.to_string(index=False))

In [ ]:
# ── 23. NULLIF to avoid division by zero ────────────────────────────────────
r = q("SELECT ROUND(SUM(GROSS_MARGIN) / NULLIF(SUM(REVENUE),0) * 100, 1) AS gm_pct FROM bw")
print('23. NULLIF:', r.iloc[0]['gm_pct'], '%')

In [ ]:
# ── 24. COALESCE for default values ──────────────────────────────────────────
r = q("SELECT PERNR, COALESCE(TERM_DATE, '9999-12-31') AS effective_end_date FROM hr LIMIT 5")
print('24. COALESCE:'); print(r.to_string(index=False))

In [ ]:
# ── 25. GROUP BY vs PARTITION BY ─────────────────────────────────────────────
sql_25 = (
    "SELECT VKORG,"
    " ROUND(SUM(NETWR),0) AS grp_total,"
    " ROUND(SUM(NETWR) OVER (PARTITION BY VKORG),0) AS part_total"
    " FROM sales LIMIT 5"
)
r = q(sql_25)
print('25. GROUP BY collapses; PARTITION BY keeps rows:')
print(r.to_string(index=False))

In [ ]:
# ── 26. HAVING vs WHERE ──────────────────────────────────────────────────────
# WHERE filters BEFORE aggregation; HAVING filters AFTER
r = q("SELECT VKORG, ROUND(SUM(NETWR),0) AS total FROM sales WHERE STATUS='OPEN' GROUP BY VKORG HAVING SUM(NETWR) > 50000")
print('26. HAVING filters groups after aggregation:'); print(r.to_string(index=False))

In [ ]:
# ── 27. INNER JOIN vs LEFT JOIN vs ANTI-JOIN ─────────────────────────────────
inner = q("SELECT COUNT(*) AS n FROM sales s JOIN mat m ON s.MATNR=m.MATNR").iloc[0]['n']
left  = q("SELECT COUNT(*) AS n FROM sales s LEFT JOIN mat m ON s.MATNR=m.MATNR").iloc[0]['n']
anti  = q("SELECT COUNT(*) AS n FROM sales s LEFT JOIN mat m ON s.MATNR=m.MATNR WHERE m.MATNR IS NULL").iloc[0]['n']
print('27. INNER:', inner, ' LEFT:', left, ' ANTI:', anti)

In [ ]:
# ── 28. EXISTS vs IN: performance difference ─────────────────────────────────
# IN: evaluates all rows in subquery first
r_in = q("SELECT COUNT(*) AS n FROM sales WHERE MATNR IN (SELECT MATNR FROM mat)").iloc[0]['n']
# EXISTS: stops as soon as first match found (faster for large subqueries)
r_ex = q("SELECT COUNT(*) AS n FROM sales s WHERE EXISTS (SELECT 1 FROM mat m WHERE m.MATNR=s.MATNR)").iloc[0]['n']
print('28. IN:', r_in, '  EXISTS:', r_ex, '  (same result, EXISTS often faster)')

In [ ]:
# ── 29. Window function: ROW_NUMBER vs RANK vs DENSE_RANK ────────────────────
sql_29 = ('SELECT SALARY,' ' ROW_NUMBER() OVER (ORDER BY SALARY DESC) AS row_num,' ' RANK() OVER (ORDER BY SALARY DESC) AS rank_,' ' DENSE_RANK() OVER (ORDER BY SALARY DESC) AS dense_rank_' ' FROM hr WHERE TERM_DATE IS NULL ORDER BY SALARY DESC LIMIT 8')
r = q(sql_29)
print('29. ROW_NUMBER vs RANK vs DENSE_RANK:')
print(r.to_string(index=False))
print('  Key: RANK skips numbers after ties; DENSE_RANK does not.')

In [ ]:
# ── 30. LAG and LEAD: accessing adjacent rows ────────────────────────────────
sql_30 = ('WITH monthly AS (SELECT CAL_YEAR_MONTH, ROUND(SUM(REVENUE),0) AS rev FROM bw GROUP BY CAL_YEAR_MONTH) ''SELECT CAL_YEAR_MONTH, rev, LAG(rev) OVER (ORDER BY CAL_YEAR_MONTH) AS prev_month, '"LEAD(rev) OVER (ORDER BY CAL_YEAR_MONTH) AS next_month FROM monthly LIMIT 5")
r = q(sql_30)
print('30. LAG/LEAD:'); print(r.to_string(index=False))

In [ ]:
# ── 31. CASE WHEN: inline conditional logic ──────────────────────────────────
sql_31 = ("SELECT KOSTL, ROUND(SUM(ACTUAL_AMT-PLAN_AMT),0) AS variance,"
          " CASE WHEN SUM(ACTUAL_AMT-PLAN_AMT)>10000 THEN 'SIGNIFICANT OVER'"
          " WHEN SUM(ACTUAL_AMT-PLAN_AMT)>0 THEN 'SLIGHT OVER'"
          " WHEN SUM(ACTUAL_AMT-PLAN_AMT)<-10000 THEN 'SIGNIFICANT UNDER'"
          " ELSE 'SLIGHT UNDER' END AS variance_bucket"
          " FROM cc WHERE GJAHR=2024 GROUP BY KOSTL LIMIT 8")
r = q(sql_31)
print('31. CASE WHEN:'); print(r.to_string(index=False))

In [ ]:
# ── 32. Subquery vs CTE: readability trade-off ───────────────────────────────
# Subquery (harder to read, can't be reused)
r_sub = q("SELECT VKORG, COUNT(*) AS n FROM (SELECT VKORG FROM sales WHERE STATUS='OPEN' AND NETWR>1000) sub GROUP BY VKORG")
# CTE (readable, reusable within the query)
r_cte = q("WITH open_big AS (SELECT VKORG FROM sales WHERE STATUS='OPEN' AND NETWR>1000) SELECT VKORG, COUNT(*) AS n FROM open_big GROUP BY VKORG")
print('32. Subquery vs CTE — same result:', (r_sub.equals(r_cte)))

In [ ]:
# ── 33. UNION vs UNION ALL ────────────────────────────────────────────────────
r_union = q("SELECT VKORG FROM sales WHERE STATUS='OPEN' UNION SELECT VKORG FROM sales WHERE NETWR>10000")
r_union_all = q("SELECT VKORG FROM sales WHERE STATUS='OPEN' UNION ALL SELECT VKORG FROM sales WHERE NETWR>10000")
print('33. UNION (dedup):', len(r_union), '  UNION ALL (keeps dups):', len(r_union_all))

In [ ]:
# ── 34. String functions: SQLite vs Standard SQL ─────────────────────────────
r = q("SELECT SUBSTR(MATNR,5) AS mat_num, UPPER(MAKTX) AS desc_upper, LENGTH(MATNR) AS len, REPLACE(MATNR,'MAT-','') AS clean_id FROM mat LIMIT 3")
print('34. String functions:'); print(r.to_string(index=False))

In [ ]:
# ── 35. Date functions in SQLite ─────────────────────────────────────────────
r = q("SELECT ERDAT, STRFTIME('%Y',ERDAT) AS year, STRFTIME('%m',ERDAT) AS month, STRFTIME('%Y-%m',ERDAT) AS year_month, CAST(JULIANDAY('2026-04-01')-JULIANDAY(ERDAT) AS INT) AS days_ago FROM sales LIMIT 3")
print('35. Date functions:'); print(r.to_string(index=False))

In [ ]:
# ── 36. Self-join: compare rows within same table ────────────────────────────
sql_36 = ('SELECT a.PERNR, a.ENAME, a.ORGEH, a.SALARY,'' b.ENAME AS colleague, b.SALARY AS colleague_salary'' FROM hr a JOIN hr b ON a.ORGEH=b.ORGEH AND a.PERNR < b.PERNR'' WHERE ABS(a.SALARY-b.SALARY)<5000 LIMIT 5')
r = q(sql_36)
print('36. Self-join (colleagues with similar salary):'); print(r.to_string(index=False))

In [ ]:
# ── 37. Aggregate window function: running total ─────────────────────────────
sql_37 = ('SELECT KOSTL, PERIOD, ROUND(SUM(ACTUAL_AMT),0) AS monthly,'' ROUND(SUM(SUM(ACTUAL_AMT)) OVER (PARTITION BY KOSTL ORDER BY PERIOD'' ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),0) AS ytd'' FROM cc WHERE GJAHR=2024 GROUP BY KOSTL, PERIOD ORDER BY KOSTL, PERIOD LIMIT 8')
r = q(sql_37)
print('37. Running YTD:'); print(r.to_string(index=False))

In [ ]:
# ── 38. DISTINCT in COUNT vs on full rows ────────────────────────────────────
r = q("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT KUNNR) AS unique_customers, COUNT(DISTINCT MATNR) AS unique_materials, COUNT(DISTINCT VBELN) AS unique_orders FROM sales")
print('38. COUNT vs COUNT DISTINCT:'); print(r.to_string(index=False))

In [ ]:
# ── 39. PERCENT_RANK vs CUME_DIST ────────────────────────────────────────────
sql_39 = ('SELECT PERNR, SALARY,'' ROUND(PERCENT_RANK() OVER (ORDER BY SALARY)*100,1) AS pct_rank,'' ROUND(CUME_DIST() OVER (ORDER BY SALARY)*100,1) AS cume_dist'' FROM hr WHERE TERM_DATE IS NULL ORDER BY SALARY DESC LIMIT 8')
r = q(sql_39)
print('39. PERCENT_RANK vs CUME_DIST:')
print(r.to_string(index=False))
print('  Key: PERCENT_RANK = rank-1 / count-1; CUME_DIST = ranks at or below / count')

In [ ]:
# ── 40. WITH RECURSIVE: quick example ───────────────────────────────────────
sql_40 = 'WITH RECURSIVE counter(n) AS (SELECT 1 UNION ALL SELECT n+1 FROM counter WHERE n<5) SELECT n FROM counter'
r = q(sql_40)
print('40. Recursive CTE (counting):', r['n'].tolist())

## Self-Assessment: Your 3 Weakest Areas
In a new markdown cell below, identify your 3 weakest concepts from the 40 above. Be specific:
- Not "SQL window functions" — write "PARTITION BY ORDER BY vs GROUP BY — I mixed them up in #25"
- Not "pandas merges" — write "join fanout (#5) — I didn't realise SUM would be wrong after joining on non-unique keys"

These 3 items are your study targets before any interview.
